# 06 — Memory does not account for the belief shift

Recall model (negativity bias, valence × condition, decline) and whether recall predicts the belief, affective, and social measures.

**Expected:** negativity bias β=+0.43, p<.001; valence×condition β=+0.58, p=.001; decline β=−0.16, p=.066. Memory→variables (amount, valence): belief drift both n.s. (amount β=−0.41, p=.52; valence β=−0.58, p=.57); connectedness n.s.; emotion amount β=−1.42, p=.07.

Note: the prevalence of encoded Cognitive Units *within the imagination task* (mean 0.025) is derived from the conversation transcripts and is not reproducible from the shared aggregates (see the transcript-coding materials / SI).

In [1]:
import pandas as pd, numpy as np, warnings
from scipy import stats
import statsmodels.formula.api as smf
warnings.filterwarnings('ignore')
DATA = '../data'
def load_belief(): return pd.read_excel(f'{DATA}/belief_items.xlsx')
def load_subject(): return pd.read_excel(f'{DATA}/subject_measures.xlsx')
def load_group(): return pd.read_csv(f'{DATA}/group_conversation.csv')
def rep(m,t):
    ci=m.conf_int().loc[t]
    return f'b={m.params[t]:+.3f} 95%CI[{ci[0]:+.3f}, {ci[1]:+.3f}] p={m.pvalues[t]:.4f}'
def corr_ci(x,y):
    r,p=stats.pearsonr(x,y); n=len(x); z=np.arctanh(r); se=1/np.sqrt(n-3)
    return r,np.tanh(z-1.96*se),np.tanh(z+1.96*se),p,n


In [2]:
s=load_subject(); d=s[s.in_memory==1].copy() if 'in_memory' in s else s.copy()
d=d.dropna(subset=['mem_CUpos_first'])
rows=[]
for _,r in d.iterrows():
    exp=int(r.condition=='collaborative')
    for cu,post,neg in [(r.mem_CUpos_first,0,0),(r.mem_CUneg_first,0,1),(r.mem_CUpos_post,1,0),(r.mem_CUneg_post,1,1)]:
        rows.append(dict(pid=r.participant,post=post,neg=neg,exp=exp,cu=cu))
L=pd.DataFrame(rows); m=smf.mixedlm('cu ~ neg + post + exp + neg:exp', L, groups=L.pid).fit()
print('N =',d.participant.nunique())
print(f'  mean CUs recalled: negative M={L[L.neg==1].cu.mean():.2f}, positive M={L[L.neg==0].cu.mean():.2f}')
print('negativity bias:', rep(m,'neg')); print('valence x condition:', rep(m,'neg:exp')); print('decline:', rep(m,'post'))


N = 125
  mean CUs recalled: negative M=2.96, positive M=2.27
negativity bias: b=+0.428 95%CI[+0.198, +0.657] p=0.0003
valence x condition: b=+0.581 95%CI[+0.239, +0.924] p=0.0009
decline: b=-0.160 95%CI[-0.330, +0.010] p=0.0658


In [3]:
# does recall predict the other measures?
d=s.copy()
d['mem_amount']=(d.mem_total_first+d.mem_total_post)/2
d['mem_valence']=((d.mem_CUneg_first-d.mem_CUpos_first)+(d.mem_CUneg_post-d.mem_CUpos_post))/2
def analyze(df,outcome,label):
    sub=df[[outcome,'mem_amount','mem_valence','group']].dropna()
    mm=smf.mixedlm(f'{outcome} ~ mem_amount + mem_valence', sub, groups=sub.group).fit(reml=False)
    print(f'{label:14s} N={len(sub)}  amount {rep(mm,"mem_amount")} | valence {rep(mm,"mem_valence")}')
analyze(d,'bel_related_change','belief drift')
analyze(d[d.condition=='collaborative'],'conn_mean','connectedness')
analyze(d,'emotion_change','emotion change')


belief drift   N=125  amount b=-0.411 95%CI[-1.665, +0.843] p=0.5204 | valence b=-0.577 95%CI[-2.554, +1.400] p=0.5673
connectedness  N=52  amount b=+0.029 95%CI[-0.060, +0.118] p=0.5204 | valence b=-0.097 95%CI[-0.273, +0.079] p=0.2792
emotion change N=125  amount b=-1.419 95%CI[-2.938, +0.099] p=0.0669 | valence b=+0.687 95%CI[-1.700, +3.075] p=0.5727
